# Resume Protein ProGen Pretraining

This notebook resumes a checkpoint produced by `03_pretrain_protein_from_scratch.ipynb`. It keeps the same protein tokenizer, model config, optimizer state, and corpus path from the checkpoint metadata.

In [ ]:
import json
import math
import sys
from pathlib import Path

import torch

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "libs").is_dir():
            return path
    raise RuntimeError("Could not locate project root from the current notebook directory.")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from libs.core import (
    LEGACY_PROTEIN_MODEL_FAMILY,
    MDCDecoderModel,
    MDCModelConfig,
    PROGEN_PROTEIN_MODEL_FAMILY,
    compute_mdc_causal_lm_loss,
    create_protein_lm_dataloader,
    create_streaming_protein_lm_dataloader,
    discover_protein_train_text_paths,
    evaluate_mdc_causal_lm_batch_loss,
    extract_protein_backbone_config,
    generate_protein_text,
    is_supported_protein_checkpoint_family,
    load_protein_corpus_text_parts,
    load_protein_pretrain_checkpoint,
    save_protein_pretrain_checkpoint,
    split_protein_corpus_text,
)
from libs.data.training.tokenizer import SequenceTokenizer

PROJECT_ROOT

In [ ]:
CHECKPOINT_DIR = PROJECT_ROOT / "data" / "checkpoints" / "protein_from_scratch"
RESUME_CHECKPOINT_PATH = CHECKPOINT_DIR / "checkpoint_last.pt"
OUTPUT_CHECKPOINT_PATH = CHECKPOINT_DIR / "checkpoint_last.pt"
BEST_CHECKPOINT_PATH = CHECKPOINT_DIR / "checkpoint_best.pt"

TRAIN_TEXT_PATH = PROJECT_ROOT / "data" / "compiled" / "refseq_bacteria_protein" / "train.txt"
TRAIN_PART_GLOB = "train_part_*.txt"
PREFER_LOCAL_TRAIN_PARTS = True
STREAM_LOCAL_TRAIN_PARTS = True
MINIO_TRAIN_PARTS_PREFIX_URI = ""
MINIO_TRAIN_PART_URIS = ()
TRAIN_PART_CACHE_DIR = PROJECT_ROOT / "data" / "cache" / "protein_train_parts"
KEEP_DOWNLOADED_TRAIN_PARTS = False

NUM_EPOCHS = 1
TRAIN_RATIO = 0.9
BATCH_SIZE = 2
STRIDE = None
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
GRAD_CLIP_NORM = 1.0
EVAL_FREQ = 50
EVAL_BATCHES = 10

if not RESUME_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Missing resume checkpoint: {RESUME_CHECKPOINT_PATH}")

checkpoint_meta = torch.load(RESUME_CHECKPOINT_PATH, map_location="cpu")
if not is_supported_protein_checkpoint_family(checkpoint_meta.get("model_family")):
    raise ValueError(
        "This resume notebook expects a protein pretrain checkpoint saved by the stage-2 flow. "
        f"Received: {checkpoint_meta.get('model_family')!r}"
    )

checkpoint_meta.keys()

In [ ]:
tokenizer_map_path = Path(checkpoint_meta["tokenizer_map_path"])
if not tokenizer_map_path.exists():
    tokenizer_map_path.parent.mkdir(parents=True, exist_ok=True)
    tokenizer_map_path.write_text(
        json.dumps(checkpoint_meta["tokenizer_map"], ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )

tokenizer = SequenceTokenizer.load_map(tokenizer_map_path)
checkpoint_corpus_files = tuple(Path(path) for path in checkpoint_meta.get("corpus_files", ()) if Path(path).exists())
train_text_path = Path(checkpoint_meta.get("corpus_file") or TRAIN_TEXT_PATH)
if checkpoint_corpus_files:
    local_train_text_paths = checkpoint_corpus_files
elif train_text_path.exists() or any(train_text_path.parent.glob(TRAIN_PART_GLOB)):
    local_train_text_paths = discover_protein_train_text_paths(
        train_text_path,
        part_glob=TRAIN_PART_GLOB,
        prefer_parts=PREFER_LOCAL_TRAIN_PARTS,
    )
else:
    local_train_text_paths = ()

use_minio_train_parts = bool(MINIO_TRAIN_PARTS_PREFIX_URI or MINIO_TRAIN_PART_URIS)
corpus_text = load_protein_corpus_text_parts(local_train_text_paths) if local_train_text_paths else ""
if not corpus_text and not use_minio_train_parts:
    raise ValueError("A local train.txt or train_part_*.txt corpus is required unless MinIO train parts are configured.")
train_text, val_text = split_protein_corpus_text(corpus_text, train_ratio=TRAIN_RATIO) if corpus_text else ("", "")

model_config_payload = dict(checkpoint_meta["model_config"])
if model_config_payload.get("layer_types") is not None:
    model_config_payload["layer_types"] = tuple(model_config_payload["layer_types"])
model_config = MDCModelConfig(**model_config_payload)
context_length = int(model_config.context_length)
stride = STRIDE or max(1, context_length // 2)

if use_minio_train_parts:
    train_loader = create_streaming_protein_lm_dataloader(
        tokenizer,
        prefix_uri=MINIO_TRAIN_PARTS_PREFIX_URI or None,
        part_uris=MINIO_TRAIN_PART_URIS or None,
        cache_dir=TRAIN_PART_CACHE_DIR,
        keep_downloaded_parts=KEEP_DOWNLOADED_TRAIN_PARTS,
        context_length=context_length,
        stride=stride,
        batch_size=BATCH_SIZE,
        shuffle_parts=True,
        shuffle_examples=True,
    )
    train_loader_description = "minio streaming"
elif STREAM_LOCAL_TRAIN_PARTS and len(local_train_text_paths) > 1:
    train_loader = create_streaming_protein_lm_dataloader(
        tokenizer,
        part_paths=local_train_text_paths,
        context_length=context_length,
        stride=stride,
        batch_size=BATCH_SIZE,
        shuffle_parts=True,
        shuffle_examples=True,
    )
    train_loader_description = "local part streaming"
else:
    train_loader = create_protein_lm_dataloader(
        train_text,
        tokenizer,
        context_length=context_length,
        stride=stride,
        batch_size=BATCH_SIZE,
        shuffle=True,
    )
    train_loader_description = "in-memory split"

val_loader = (
    create_protein_lm_dataloader(
        val_text,
        tokenizer,
        context_length=context_length,
        stride=stride,
        batch_size=BATCH_SIZE,
        shuffle=False,
    )
    if val_text
    else None
)

def dataloader_size(loader):
    if loader is None:
        return "disabled"
    try:
        return len(loader)
    except TypeError:
        return "streaming"

print(f"Corpus: {train_text_path}")
print(f"Local corpus files: {len(local_train_text_paths)}")
print(f"Train loader: {train_loader_description}")
print(f"Tokenizer: {tokenizer_map_path} vocab={tokenizer.vocab_size}")
print(f"Context length: {context_length} stride={stride}")
print(f"Checkpoint family: {checkpoint_meta.get('model_family') or LEGACY_PROTEIN_MODEL_FAMILY}")
print(f"Batches: train={dataloader_size(train_loader)} val={dataloader_size(val_loader)}")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MDCDecoderModel(model_config).to(device)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
checkpoint = load_protein_pretrain_checkpoint(
    RESUME_CHECKPOINT_PATH,
    model=model,
    optimizer=optimizer,
    map_location=device,
)

start_epoch = int(checkpoint.get("epoch", 0))
global_step = int(checkpoint.get("global_step", 0))
tokens_seen = int(checkpoint.get("tokens_seen", 0))
train_losses = list(checkpoint.get("train_losses", []))
val_losses = list(checkpoint.get("val_losses", []))
best_val_loss = checkpoint.get("best_val_loss")
best_val_loss = math.inf if best_val_loss is None else float(best_val_loss)

print(f"Resumed from epoch={start_epoch} step={global_step} tokens={tokens_seen:,} on {device}")

In [ ]:
metrics_path = CHECKPOINT_DIR / "resume_training_metrics.jsonl"

def move_batch(batch, device):
    return type(batch)(
        input_ids=batch.input_ids.to(device),
        attention_mask=batch.attention_mask.to(device),
        labels=batch.labels.to(device),
    )

def append_metrics(payload):
    with metrics_path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(payload, ensure_ascii=False) + "\n")

def save_checkpoint(path, epoch, val_loss):
    return save_protein_pretrain_checkpoint(
        path,
        model=model,
        optimizer=optimizer,
        model_config=model_config,
        tokenizer=tokenizer,
        tokenizer_map_path=tokenizer_map_path,
        epoch=epoch,
        global_step=global_step,
        tokens_seen=tokens_seen,
        train_losses=train_losses,
        val_losses=val_losses,
        best_val_loss=None if math.isinf(best_val_loss) else best_val_loss,
        training_args={
            "batch_size": BATCH_SIZE,
            "context_length": context_length,
            "stride": stride,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "resumed_from": str(RESUME_CHECKPOINT_PATH.resolve()),
        },
        extra={
            "corpus_file": str(train_text_path.resolve()),
            "corpus_files": [str(path.resolve()) for path in local_train_text_paths],
            "minio_train_parts_prefix_uri": MINIO_TRAIN_PARTS_PREFIX_URI,
            "minio_train_part_uris": list(MINIO_TRAIN_PART_URIS),
            "progen_config": extract_protein_backbone_config(checkpoint),
            "last_eval_val_loss": val_loss,
        },
    )

for epoch_offset in range(1, NUM_EPOCHS + 1):
    epoch = start_epoch + epoch_offset
    model.train()
    for batch in train_loader:
        batch = move_batch(batch, device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(batch.input_ids, attn_mask=batch.attention_mask)
        loss = compute_mdc_causal_lm_loss(logits, batch.labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        optimizer.step()

        global_step += 1
        tokens_seen += int(batch.attention_mask.sum().item())

        if EVAL_FREQ > 0 and global_step % EVAL_FREQ == 0:
            train_eval_loss = evaluate_mdc_causal_lm_batch_loss(
                model, train_loader, device=device, max_batches=EVAL_BATCHES
            )
            val_loss = (
                evaluate_mdc_causal_lm_batch_loss(model, val_loader, device=device, max_batches=EVAL_BATCHES)
                if val_loader is not None
                else float("nan")
            )
            train_losses.append(train_eval_loss)
            val_losses.append(val_loss)
            append_metrics({
                "epoch": epoch,
                "global_step": global_step,
                "tokens_seen": tokens_seen,
                "train_loss": train_eval_loss,
                "val_loss": val_loss,
            })
            if val_loader is not None and val_loss < best_val_loss:
                best_val_loss = val_loss
                save_checkpoint(BEST_CHECKPOINT_PATH, epoch, val_loss)
            print(f"epoch={epoch} step={global_step} train={train_eval_loss:.4f} val={val_loss:.4f}")

    val_loss = (
        evaluate_mdc_causal_lm_batch_loss(model, val_loader, device=device, max_batches=EVAL_BATCHES)
        if val_loader is not None
        else float("nan")
    )
    save_checkpoint(OUTPUT_CHECKPOINT_PATH, epoch, val_loss)

print(f"Saved resumed checkpoint: {OUTPUT_CHECKPOINT_PATH}")

In [ ]:
sample = generate_protein_text(
    model,
    tokenizer,
    prompt="<|protein|>",
    device=device,
    max_new_tokens=80,
    context_length=context_length,
)
sample